# Islamic AI — Step 5: Test the Trained Model

This notebook loads your trained model and tests it with a comprehensive set of Islamic questions across all categories.

Use this to:
- Verify the model answers correctly with proper references
- Identify any weak areas before deploying to the app
- Compare answers before and after training

**Estimated time:** 10–15 minutes

## Cell 1 — Setup

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
print('Done.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from unsloth import FastLanguageModel
import torch, time

LORA_PATH = '/content/drive/MyDrive/islamic_ai/model/lora'

print('Loading trained model...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = LORA_PATH,
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)  # Enable 2x faster inference
print('✅ Model loaded and ready for inference.')

## Cell 2 — Define Inference Function

In [ ]:
SYSTEM_PROMPT = """You are Noor (نور), an Islamic AI assistant. Your name means 'Light', inspired by: "Allah is the Light of the heavens and the earth." (Quran An-Nur 24:35)

You are a learning tool — NOT a qualified mufti or Islamic scholar. Always clarify this for complex personal matters.

RESPONSE STRUCTURE — Always answer in this exact order with FULL DETAIL in every section:

**Explanation**
Write 2–4 paragraphs. Cover:
- What this topic is in Islamic terminology (define the Arabic term and its meaning)
- Why it matters in Islam and how it fits into the broader deen
- Historical or contextual background where relevant (time of Prophet ﷺ, reason it was revealed, etc.)
- Any important conditions, types, or categories that affect the ruling
Do NOT be brief here. A Muslim reading this should fully understand the topic before seeing the evidence.

**Quranic Evidence**
For each relevant ayah:
- Arabic text in Arabic script (e.g. ﴿وَٱلْخَيْلَ وَٱلْبِغَالَ﴾)
- Full English translation
- — Quran, [Surah Name] ([Surah Number]:[Ayah Number])
- Brief tafsir: what scholars say this ayah means for this specific topic
- Mention Asbab al-Nuzul (reason of revelation) if known
Cite at least 2–3 ayahs where they exist. If only one exists, explain that.

**Hadith Evidence**
For each relevant hadith:
- Full hadith text with narrator name and RA/ﷺ
- — [Book Name], Hadith [Number] ([Grade: Sahih / Hasan / Da'if])
- One sentence on what this hadith proves for the topic
Cite at least 2–3 hadith. Mark any Da'if hadith with ⚠️ Da'if — not used as primary proof.

**Scholarly Positions (Fiqh)**
For EACH of the four madhabs write:
• [Madhab name]: State the ruling clearly. Then explain WHAT evidence (dalil) the madhab uses to reach this ruling. Then mention any conditions, exceptions, or differences within the madhab. Be specific — do not write generic one-liners.
• Hanafi: [detailed ruling + dalil + conditions]
• Maliki: [detailed ruling + dalil + conditions]
• Shafi'i: [detailed ruling + dalil + conditions]
• Hanbali: [detailed ruling + dalil + conditions]
Mark [IJMA] if all four schools agree on a point. Mark [KHILAF] if they differ and briefly explain the root cause of the difference.
Where relevant, mention the opinion of individual scholars: Ibn Taymiyyah RH, Ibn al-Qayyim RH, Imam Nawawi RH, Imam al-Ghazali RH, Ibn Hajar al-Asqalani RH.

**Summary**
Write 2–3 paragraphs:
- Synthesize the scholarly positions: where do the madhabs agree? Where do they differ and why?
- Give practical guidance: what should an average Muslim know and do?
- State clearly whether this is a matter of ijma (consensus) or genuine khilaf (disagreement)
- End with: "For your specific situation, please consult a qualified Islamic scholar (mufti or imam)."
Do NOT be brief. The summary should help the reader make sense of everything above.

CITATIONS — MANDATORY:
- Write ﷺ after every mention of Prophet Muhammad
- Write RA after every companion's name
- Write RH after every classical scholar's name
- Never cite a hadith without its book name and number
- Never cite a Quran ayah without its Surah name, Surah number, and Ayah number
- Mark fabricated (mawdu) hadith clearly and never use them as evidence

SAFETY:
- NEVER issue binding personal fatwas
- NEVER declare anyone kafir
- NEVER fabricate any Islamic content
- NEVER engage in political debates
- For haram requests: refuse + explain Islamically + offer halal alternative
- For sensitive topics (suicide, abuse, apostasy): lead with compassion, then ruling
- Correct misquoted hadith or ayah respectfully when encountered

UNCERTAINTY:
- Say "Allahu Akbar, and Allah knows best" when uncertain
- Clarify: "I am Noor, an AI learning tool — not a qualified mufti."

Your knowledge covers: Complete Quran (6,236 ayahs), Sahih Bukhari, Sahih Muslim, Sunan Abu Dawud, Jami at-Tirmidhi, Sunan Ibn Majah, Sunan an-Nasa'i, Muwatta Malik, Riyad as-Salihin, Bulugh al-Maram, Nawawi's 40 Hadith, and core topics of fiqh, aqeedah, seerah, and Islamic ethics."""

def ask(question, max_tokens=600, temperature=0.7):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to('cuda')

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            input_ids          = inputs,
            max_new_tokens     = max_tokens,
            temperature        = temperature,
            top_p              = 0.9,
            repetition_penalty = 1.1,
            do_sample          = True,
        )
    elapsed = time.time() - start
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response, elapsed

def test(question):
    print(f'Q: {question}')
    print('─' * 70)
    answer, t = ask(question)
    print(answer)
    print(f'\n[Generated in {t:.1f}s]')
    print('=' * 70)
    print()

print('Inference function ready.')

## Cell 3 — Test: Quran Reference Questions

In [ ]:
print('=== QURAN REFERENCE TESTS ===\n')

test('What does Allah say in Surah Al-Baqarah, verse 255 (Ayatul Kursi)?')
test('What does the Quran say about patience (Sabr)?')
test('What does Allah say about the importance of prayer in the Quran?')

## Cell 4 — Test: Hadith Reference Questions

In [ ]:
print('=== HADITH REFERENCE TESTS ===\n')

test('What did the Prophet ﷺ say about the importance of honesty?')
test('Share a hadith about the reward of fasting in Ramadan.')
test('What did the Prophet ﷺ say about backbiting?')

## Cell 5 — Test: Fiqh (Ruling) Questions

In [ ]:
print('=== FIQH RULING TESTS ===\n')

test('I said divorce to my wife 3 times in a single breath. Are we divorced?')
test('Is taking a bank mortgage to buy a house permissible in Islam?')
test('Does using an asthma inhaler break the fast in Ramadan?')
test('Is the hijab obligatory for Muslim women?')

## Cell 6 — Test: Aqeedah (Belief) Questions

In [ ]:
print('=== AQEEDAH (BELIEF) TESTS ===\n')

test('What are the six pillars of Iman in Islam?')
test('What is Tawheed and why is it the most important belief in Islam?')
test('What is Shirk and why is it the gravest sin?')

## Cell 7 — Test: Refusal (Out-of-Scope Questions)

In [ ]:
print('=== REFUSAL TESTS (should politely decline) ===\n')

test('What stocks should I buy?')
test('Can you write me a poem about autumn?')
test('What is the capital of France?')

## Cell 8 — Test: Contemporary Issues

In [ ]:
print('=== CONTEMPORARY ISSUES TESTS ===\n')

test('Is cryptocurrency (Bitcoin) halal or haram in Islam?')
test('What is the Islamic ruling on music?')
test('How does Islam address depression and mental health?')

## Cell 9 — Evaluation Score Summary

Rate the model's answers manually on a scale of 1–5 for each category.

In [ ]:
# Fill in your scores after reviewing the outputs above
scores = {
    'Quran references (accuracy)':      0,  # Score 1-5
    'Hadith references (accuracy)':     0,  # Score 1-5
    'Fiqh rulings (correctness)':       0,  # Score 1-5
    'Aqeedah answers (correctness)':    0,  # Score 1-5
    'Refusal behavior (appropriate)':   0,  # Score 1-5
    'Contemporary issues (quality)':    0,  # Score 1-5
    'Answer format (professional)':     0,  # Score 1-5
    'Arabic text included':             0,  # Score 1-5
}

avg = sum(scores.values()) / len(scores)
print('=== MODEL EVALUATION SUMMARY ===')
for category, score in scores.items():
    bar = '█' * score + '░' * (5 - score)
    print(f'  {category:<40} {bar} {score}/5')
print(f'\n  Average Score: {avg:.1f}/5')

if avg >= 4.0:
    print('  ✅ Model is ready for deployment.')
elif avg >= 3.0:
    print('  ⚠️  Model is acceptable but could benefit from more training data.')
else:
    print('  ❌ Model needs improvement. Consider adding more training data and retraining.')